# Portfolio Feature Engineering from hey_productos.csv

This notebook implements feature engineering for portfolio analysis based on the `hey_productos.csv` dataset.
Features are computed per user (`user_id`).

## Features to Implement
- `product_diversity_score`: Count of distinct `tipo_producto` per user
- `has_credit`: Any of: `tarjeta_credito_hey`, `credito_personal`, `credito_auto`, `credito_nomina`
- `avg_credit_utilization`: Mean `utilizacion_pct` across all credit products
- `max_credit_limit`: Max `limite_credito` across products
- `has_investment`: `inversion_hey` presence with `saldo_actual` > 0
- `has_insurance`: `seguro_vida` or `seguro_compras` presence
- `secured_card_flag`: `tarjeta_credito_garantizada` presence
- `portfolio_age_days`: Days since first product `fecha_apertura`
- `credit_product_count`: Total number of credit-type products
- `investment_balance`: Sum of `saldo_actual` where `tipo_producto == 'inversion_hey'`

In [1]:
# Step 1: Import Libraries
import pandas as pd
import numpy as np
from datetime import datetime, date
import os

In [2]:
# Step 2: Load Data
data_path = '../Data/dataset_transacciones/hey_productos.csv'
df = pd.read_csv(data_path)
print(f"Loaded {len(df)} records")
print(df.head())

Loaded 38909 records
    producto_id    user_id        tipo_producto fecha_apertura estatus  \
0  PRD-00000001  USR-00001        cuenta_debito     2023-06-26  activo   
1  PRD-00000002  USR-00001  tarjeta_credito_hey     2022-10-16  activo   
2  PRD-00000003  USR-00002        cuenta_debito     2025-04-03  activo   
3  PRD-00000004  USR-00002  tarjeta_credito_hey     2025-07-18  activo   
4  PRD-00000005  USR-00003        cuenta_debito     2023-03-26  activo   

   limite_credito  saldo_actual  utilizacion_pct  tasa_interes_anual  \
0             NaN      80954.60              NaN                 NaN   
1        144000.0      88790.40           0.6166               35.71   
2             NaN      20712.54              NaN                 NaN   
3         22000.0       6122.60           0.2783               34.08   
4             NaN       3454.65              NaN                 NaN   

   plazo_meses  monto_mensualidad fecha_ultimo_movimiento  es_dato_sintetico  
0          NaN        

In [3]:
# Step 3: Preprocess Data
# Convert date columns
df['fecha_apertura'] = pd.to_datetime(df['fecha_apertura'])
df['fecha_ultimo_movimiento'] = pd.to_datetime(df['fecha_ultimo_movimiento'])

# Define constants
CREDIT_PRODUCTS = ['tarjeta_credito_hey', 'tarjeta_credito_garantizada', 'tarjeta_credito_negocios', 
                   'credito_personal', 'credito_auto', 'credito_nomina']
HAS_CREDIT_PRODUCTS = ['tarjeta_credito_hey', 'credito_personal', 'credito_auto', 'credito_nomina']
INSURANCE_PRODUCTS = ['seguro_vida', 'seguro_compras']
CURRENT_DATE = date(2026, 4, 25)

print("Data preprocessing complete")

Data preprocessing complete


In [4]:
# Step 4: Group Data by user_id
grouped = df.groupby('user_id')
print(f"Number of unique users: {len(grouped)}")

Number of unique users: 15025


In [5]:
# Step 5: Compute product_diversity_score
product_diversity_score = grouped['tipo_producto'].nunique()
print("product_diversity_score computed")
print(product_diversity_score.head())

product_diversity_score computed
user_id
USR-00001    2
USR-00002    2
USR-00003    2
USR-00004    3
USR-00005    2
Name: tipo_producto, dtype: int64


In [6]:
# Step 6: Compute has_credit
def has_credit(products):
    return any(p in HAS_CREDIT_PRODUCTS for p in products)

has_credit_flag = grouped['tipo_producto'].apply(has_credit)
print("has_credit computed")
print(has_credit_flag.head())

has_credit computed
user_id
USR-00001    True
USR-00002    True
USR-00003    True
USR-00004    True
USR-00005    True
Name: tipo_producto, dtype: bool


In [7]:
# Step 7: Compute avg_credit_utilization
def avg_credit_util(group):
    credit_rows = group[group['tipo_producto'].isin(CREDIT_PRODUCTS)]
    if not credit_rows.empty:
        return credit_rows['utilizacion_pct'].mean()
    return np.nan

avg_credit_utilization = grouped.apply(avg_credit_util)
print("avg_credit_utilization computed")
print(avg_credit_utilization.head())

avg_credit_utilization computed
user_id
USR-00001    0.6166
USR-00002    0.2783
USR-00003    0.3091
USR-00004    0.3538
USR-00005    0.6326
dtype: float64


/tmp/ipykernel_131237/3601585064.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  avg_credit_utilization = grouped.apply(avg_credit_util)


In [8]:
# Step 8: Compute max_credit_limit
max_credit_limit = grouped['limite_credito'].max()
print("max_credit_limit computed")
print(max_credit_limit.head())

max_credit_limit computed
user_id
USR-00001    144000.0
USR-00002     22000.0
USR-00003      5000.0
USR-00004     67000.0
USR-00005    136000.0
Name: limite_credito, dtype: float64


In [9]:
# Step 9: Compute has_investment
def has_investment(group):
    inv_rows = group[(group['tipo_producto'] == 'inversion_hey') & (group['saldo_actual'] > 0)]
    return not inv_rows.empty

has_investment_flag = grouped.apply(has_investment)
print("has_investment computed")
print(has_investment_flag.head())

has_investment computed
user_id
USR-00001    False
USR-00002    False
USR-00003    False
USR-00004     True
USR-00005    False
dtype: bool


/tmp/ipykernel_131237/3866309592.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  has_investment_flag = grouped.apply(has_investment)


In [10]:
# Step 10: Compute has_insurance
def has_insurance(products):
    return any(p in INSURANCE_PRODUCTS for p in products)

has_insurance_flag = grouped['tipo_producto'].apply(has_insurance)
print("has_insurance computed")
print(has_insurance_flag.head())

has_insurance computed
user_id
USR-00001    False
USR-00002    False
USR-00003    False
USR-00004    False
USR-00005    False
Name: tipo_producto, dtype: bool


In [11]:
# Step 11: Compute secured_card_flag
def has_secured_card(products):
    return 'tarjeta_credito_garantizada' in products

secured_card_flag = grouped['tipo_producto'].apply(has_secured_card)
print("secured_card_flag computed")
print(secured_card_flag.head())

secured_card_flag computed
user_id
USR-00001    False
USR-00002    False
USR-00003    False
USR-00004    False
USR-00005    False
Name: tipo_producto, dtype: bool


In [12]:
# Step 12: Compute portfolio_age_days
portfolio_age_days = grouped['fecha_apertura'].min().apply(lambda x: (CURRENT_DATE - x.date()).days)
print("portfolio_age_days computed")
print(portfolio_age_days.head())

portfolio_age_days computed
user_id
USR-00001    1287
USR-00002     387
USR-00003    1126
USR-00004    1118
USR-00005     746
Name: fecha_apertura, dtype: int64


In [13]:
# Step 13: Compute credit_product_count
def count_credit_products(products):
    return sum(1 for p in products if p in CREDIT_PRODUCTS)

credit_product_count = grouped['tipo_producto'].apply(count_credit_products)
print("credit_product_count computed")
print(credit_product_count.head())

credit_product_count computed
user_id
USR-00001    1
USR-00002    1
USR-00003    1
USR-00004    1
USR-00005    1
Name: tipo_producto, dtype: int64


In [14]:
# Step 14: Compute investment_balance
def sum_investment_balance(group):
    inv_rows = group[group['tipo_producto'] == 'inversion_hey']
    return inv_rows['saldo_actual'].sum() if not inv_rows.empty else 0

investment_balance = grouped.apply(sum_investment_balance)
print("investment_balance computed")
print(investment_balance.head())

investment_balance computed
user_id
USR-00001         0.00
USR-00002         0.00
USR-00003         0.00
USR-00004    319425.53
USR-00005         0.00
dtype: float64


/tmp/ipykernel_131237/1485908864.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  investment_balance = grouped.apply(sum_investment_balance)


In [15]:
# Step 15: Assemble Final DataFrame
features_df = pd.DataFrame({
    'user_id': product_diversity_score.index,
    'product_diversity_score': product_diversity_score.values,
    'has_credit': has_credit_flag.values,
    'avg_credit_utilization': avg_credit_utilization.values,
    'max_credit_limit': max_credit_limit.values,
    'has_investment': has_investment_flag.values,
    'has_insurance': has_insurance_flag.values,
    'secured_card_flag': secured_card_flag.values,
    'portfolio_age_days': portfolio_age_days.values,
    'credit_product_count': credit_product_count.values,
    'investment_balance': investment_balance.values
})

print("Final features DataFrame created")
print(features_df.head())
print(f"Shape: {features_df.shape}")

Final features DataFrame created
     user_id  product_diversity_score  has_credit  avg_credit_utilization  \
0  USR-00001                        2        True                  0.6166   
1  USR-00002                        2        True                  0.2783   
2  USR-00003                        2        True                  0.3091   
3  USR-00004                        3        True                  0.3538   
4  USR-00005                        2        True                  0.6326   

   max_credit_limit  has_investment  has_insurance  secured_card_flag  \
0          144000.0           False          False              False   
1           22000.0           False          False              False   
2            5000.0           False          False              False   
3           67000.0            True          False              False   
4          136000.0           False          False              False   

   portfolio_age_days  credit_product_count  investment_balance  

In [16]:
# Step 16: Validation
print("Data types:")
print(features_df.dtypes)
print("\nMissing values:")
print(features_df.isnull().sum())
print("\nSummary statistics:")
print(features_df.describe())

Data types:
user_id                     object
product_diversity_score      int64
has_credit                    bool
avg_credit_utilization     float64
max_credit_limit           float64
has_investment                bool
has_insurance                 bool
secured_card_flag             bool
portfolio_age_days           int64
credit_product_count         int64
investment_balance         float64
dtype: object

Missing values:
user_id                       0
product_diversity_score       0
has_credit                    0
avg_credit_utilization     4055
max_credit_limit           4055
has_investment                0
has_insurance                 0
secured_card_flag             0
portfolio_age_days            0
credit_product_count          0
investment_balance            0
dtype: int64

Summary statistics:
       product_diversity_score  avg_credit_utilization  max_credit_limit  \
count             15025.000000            10970.000000      10970.000000   
mean                  2.589617    

In [18]:
# Step 17: Save Results (Optional)
features_df.to_csv('portfolio_features.csv', index=False)
# print("Features saved to portfolio_features.csv")

# For now, just display
features_df

,user_id,product_diversity_score,has_credit,avg_credit_utilization,max_credit_limit,has_investment,has_insurance,secured_card_flag,portfolio_age_days,credit_product_count,investment_balance
0,USR-00001,2,True,0.6166,144000.0,False,False,False,1287,1,0.00
1,USR-00002,2,True,0.2783,22000.0,False,False,False,387,1,0.00
2,USR-00003,2,True,0.3091,5000.0,False,False,False,1126,1,0.00
3,USR-00004,3,True,0.3538,67000.0,True,False,False,1118,1,319425.53
4,USR-00005,2,True,0.6326,136000.0,False,False,False,746,1,0.00
...,...,...,...,...,...,...,...,...,...,...,...
15020,USR-15021,3,True,0.1755,29000.0,True,False,False,1269,1,159277.54
15021,USR-15022,2,True,0.2361,32000.0,False,False,False,1352,1,0.00
15022,USR-15023,1,False,NaN,NaN,False,False,False,392,0,0.00
15023,USR-15024,2,False,0.2032,10000.0,False,False,False,378,1,0.00
